# Scoring Model Analysis

This notebook analyzes the distribution of scores produced by the scoring engine
across a set of test tenders.

## Purpose

- Check for systematic bias in score distributions (too many BID / REVIEW / NO BID)
- Calibrate scoring weights against expert judgment
- Identify score components that drive recommendations
- Validate that score thresholds (60, 80, 70 incumbent) are well-placed

## Setup

Run `scripts/score_all.py` before this notebook to populate `data/outcomes/` with recommendation JSONs.

In [ ]:
import sys
import json
sys.path.insert(0, '..')

from pathlib import Path
from src.utils.helpers import load_json

outcomes_dir = Path('../data/outcomes')
rec_files = list(outcomes_dir.glob('*_recommendation.json'))
print(f'Found {len(rec_files)} recommendation(s)')

recommendations = [load_json(f) for f in rec_files]
print(f'Loaded {len(recommendations)} recommendations')

## Recommendation Distribution

In [ ]:
from collections import Counter

if recommendations:
    dist = Counter(r['recommendation'] for r in recommendations)
    total = len(recommendations)
    print('Recommendation Distribution:')
    for rec, count in sorted(dist.items()):
        print(f'  {rec}: {count} ({count/total*100:.0f}%)')
    
    print()
    scores = [r['qualification_score'] for r in recommendations]
    print(f'Qualification Scores:')
    print(f'  Min:    {min(scores)}')
    print(f'  Max:    {max(scores)}')
    print(f'  Mean:   {sum(scores)/len(scores):.1f}')
    print(f'  Median: {sorted(scores)[len(scores)//2]}')
else:
    print('No recommendations yet. Run scripts/score_all.py first.')

## Score Component Breakdown

In [ ]:
if recommendations:
    score_fields = ['qualification_score', 'competitive_strength', 'incumbent_risk', 'execution_risk', 'value_score']
    
    print(f'{"Score":<25} {"Min":>6} {"Max":>6} {"Mean":>8}')
    print('-' * 50)
    for field in score_fields:
        values = [r[field] for r in recommendations if r.get(field) is not None]
        if values:
            print(f'{field:<25} {min(values):>6} {max(values):>6} {sum(values)/len(values):>8.1f}')